In [16]:
!pip install -qU \
    openai \
    langchain \
    langchain-openai \
    langchain-community \
    langchain-text-splitters \
    faiss-cpu pypdf \
    python-dotenv \
    pydantic \
    pandas \
    chromadb

## LangChain + OPenAI from scracth

# We will learn to real AI Applications using:

- python
- OpenAI
- LangChain
- Prompt Templates
- Structured outputs
- Memory
- PDF loading
- Text Splittig
- Embedding
- Vector Database
- RAG: Retrival-Augmented Genration

We will build an AI assistant that can read a PDF and answer questions from it.

Example:

> "What problem does LangChain solve?"

> "Explain prompt templates."

> "Why do we split documents?"

This RAG is useful for:

- Course assistants
- HR policy bots
- Legal document assistants
- Resume analyzers
- Finance report readers
- ESG report assistants
- Internal company knowledge bots

# 1. Why LangChain?

OpenAI model alone can answer questions.

But real applications need more than simple Q&A.

Real applications need:

- Prompt Template
- Document Reading
- Memory
- Structured JSON output
- Multi-step workflows
- Search over PDFs
- Retrieval from kowledge bases

LangChain helps us organize all these steps cleanly. 




#Load Environment Variables

We will load the API key from the `.env` file.

This keeps our code clean and secure.

## Load Environment variable

In [17]:
import os
from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")
model_name = os.getenv("MODEL", "gpt-3.5-turbo")
embedding_model_name = os.getenv("EMBEDDING_MODEL", "text-embedding-3-small")

if api_key:
    print("API key has been loaded successfully.")
if model_name:
    print("Model name has been loaded successfully.")
if embedding_model_name:
    print("Embedding model name has been loaded successfully.")

else:
    print("Failed to load API key or model names. Please check your .env file.")    

API key has been loaded successfully.
Model name has been loaded successfully.
Embedding model name has been loaded successfully.


## OpenAI Without LangChain

In [18]:
from openai import OpenAI

client = OpenAI(api_key=api_key)

response = client.responses.create(
    model=model_name,
    input="Explain Langchain in one simple sentence for a beginner."
)

print(response.output_text)

Langchain is a framework that helps developers build applications using language models by providing tools to connect, manage, and enhance their capabilities.


In [19]:
from langchain_openai import ChatOpenAI

llm =ChatOpenAI(
    model=model_name,
    temperature=0.7
)

response = llm.invoke("Explain Langchain in one simple sentence for a beginner.")

print(response.content)

Langchain is a framework that helps developers build applications powered by language models, making it easier to integrate and use AI for tasks like chatbots, data analysis, and content generation.


## Prompt Templates

Problem:

If we write prompts manually again and again, we repeat ourselves.

Example:

Explain Selenium to a beginner.  
Explain Playwright to a beginner.  
Explain LangChain to a beginner.  

Instead, we can create a reusable template.

A prompt template is like a form with blanks.

Example:

"Explain {topic} to a beginner with one real-life example."

This is our first chain.

The flow is:

User Input → Prompt Template → OpenAI Model → Output Parser → Final Answer

The pipe symbol `|` means:

"Send the output of one step to the next step."

This is called LangChain Expression Language (LCEL).

In [20]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser


prompt = ChatPromptTemplate.from_template(
    """
    You are a friendly AI trainer.

    Explain the topic: {topic}

    Rules:
    - Explain in beginner-friendly language
    - Use one real-life example
    - Give one interview-style question at the end
    """
)

chain = prompt | llm | StrOutputParser()

response = chain.invoke({"topic": "Langchain"})

print(response)

### What is Langchain?

Langchain is a framework designed to help developers build applications that use language models, like the ones from OpenAI or other similar AI systems. These applications can do things like chat with users, answer questions, or even help generate text. Think of Langchain as a toolkit that makes it easier to create smart applications that can understand and generate human language.

Imagine you want to create a virtual assistant that helps people find recipes. You can use Langchain to connect a language model that can understand questions about cooking and provide answers, suggest recipes, or explain cooking techniques. This way, instead of starting from scratch, you can leverage the power of language models with the tools Langchain provides, making your job much easier.

### Real-Life Example

Let’s say you’re a developer who wants to create a chatbot for a customer service application. Using Langchain, you can quickly set up a system where users can ask questi

## Structured Output

Normal LLM output is plain text.

But real applications often need JSON.

Example:

Instead of this:

"Rakesh is a Senior QA Automation Test Engineer with 10 years of experience."

We want this: JSON style

{
"name": "Rakesh",
"role": "Senior QA Automation Test Engineer"
"experience_years": 10,
"skills": ["Selenium, "Playwright", "Pyhon"]
}

Structured output is useful when we want to:

- Save data into database
- Send data to API
- Show data in UI
- Validate response format

In [21]:
from typing import List
from pydantic import BaseModel, Field

class StudentProfile(BaseModel):
    name: str = Field(description="Name of the person")
    role: str = Field(description="Current role")
    experience_years: int = Field(description="Total years of experience")
    skills: List[str] = Field(description="List of technical skills")

structured_llm = llm.with_structured_output(StudentProfile)

profile_text = """
Rakesh Kumar Singh is a Senior QA Automation Engineer with 10 years of experience.
He knows Selenium, Java, Playwright, Python, API Testing, and Generative AI.
"""

result = structured_llm.invoke(
    f"Extract the profile information from this text:\n{profile_text}"
)

print(result)
print("Name:", result.name)
print("Skills:", result.skills)

name='Rakesh Kumar Singh' role='Senior QA Automation Engineer' experience_years=10 skills=['Selenium', 'Java', 'Playwright', 'Python', 'API Testing', 'Generative AI']
Name: Rakesh Kumar Singh
Skills: ['Selenium', 'Java', 'Playwright', 'Python', 'API Testing', 'Generative AI']


## Load a PDF

Now we move toward a real use case.

We will load a PDF and ask questions from it.

Keep our PDF in the same folder as this notebook.

Example file name:

dotcom-secrets.pdf

## PyPDFLoader reads the PDF.

It converts every page into a Document object.

Each Document has:

- page_content
- metadata

page_content contains the text.

metadata contains information like page number and source file.

In [22]:
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader

pdf_path = Path("data/dotcom-secrets.pdf")

if pdf_path.exists():
    print(f"PDF file found at: {pdf_path}")

else: 
    print(f"PDF file not found at: {pdf_path}. Please check the path and try again.")

PDF file found at: data/dotcom-secrets.pdf


In [23]:
loader = PyPDFLoader(str(pdf_path))

documents = loader.load()

print(f"Number of documents loaded: {len(documents)}")
print(f"\nFirst document content:\n{documents[101].page_content[:5000]}")
print(f"\nMetadata of the document:\n{documents[101].metadata}")


Number of documents loaded: 274

First document content:
The Soap Opera Sequence   |   83
Email #5: Urgency and CTA. This is usually the last email in 
my Soap Opera Sequence. It’s NOT the last email I send people, 
it’s just the end of my introduction. The goal is to give the reader 
one last push to go take action right now. You do that by adding 
urgency into the equation and then using a call to action (CTA). 
Up to now, you’ve been casually using CTAs, but in this last email, 
you want to light a little ﬁre under readers. What legitimate reasons 
can you come up with that would make them need to take action  
right away?
• Your webinar starts tomorrow.
• You only have ten seats left at your event.
• You only ordered one thousand books, and most of them  
are gone.
• You’re pulling the video oﬄine.
Whatever the reason, it needs to be real. Fake urgency will backﬁre 
on you, and you’ll lose all credibility. Just think of a reason why you 
might “run out” of whatever you’re selling. 

## Text Splitting / Chunking

chunk_size=800 means each chunk will have around 800 characters.

chunk_overlap=150 means some text will repeat between chunks.

Why overlap?

Because sometimes an important answer may start in one chunk and continue in the next.

Overlap helps preserve context.

In [24]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150
)

chunks = text_splitter.split_documents(documents)

print(f"Number of chunks created: {len(chunks)}")
print(f"\nFirst chunk content:")
print(chunks[440].page_content)
print(f"\nMetadata of the first chunk:")
print(chunks[440].metadata)


print(f"\nWhich page has this content:")
print(f"Page Number: {chunks[440].metadata.get('page_label')}")

Number of chunks created: 639

First chunk content:
will attract consumers and get buyers into your Value Ladder. 
Let’s Review:  The secret to converting cold traﬃc is leveraging 
the power of free. Whatever you’re sharing in your free-plus-shipping 
oﬀer, it can’t be boring, general knowledge. It has to be unique, sexy, or 
fun—the more unique, the better. Using free-plus-shipping oﬀers is the 
fastest way to qualify your buyers and get people into your Value Ladder. 
Remember, if someone isn’t willing to pull out a credit card and pay a 
few bucks for shipping on your product, then they probably aren’t going 
to buy your other products either. 
Up Next: Now that we’ve covered all the strategy behind Value 
Ladders and how to turn those into sales funnels, would you like to see

Metadata of the first chunk:
{'producer': 'calibre (5.22.1) [https://calibre-ebook.com]', 'creator': 'calibre (5.22.1) [https://calibre-ebook.com]', 'creationdate': '2015-01-06T17:12:30-07:00', 'author': 'Rus

## Embeddings

Embeddings convert text into numbers.

But these numbers capture meaning.

Example:

"car" and "vehicle" will be close in meaning.

"car" and "banana" will be far in meaning.

Embeddings help us search by meaning, not only by exact keyword.

Do not worry about the numbers.

The important point is:

Text has been converted into a mathematical representation.

Now we can compare meaning mathematically.

Do not worry about the numbers.

The important point over here is:

### Now we can compare the meaning mathematicaly

In [25]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(
    model=embedding_model_name,
)

#this is the text what user has asked we want 
# to convert into a vector using the embedding model so that it sits around the answer in vector database
sample_text = "What is the main topic of the document?"

sample_vector = embeddings.embed_query(sample_text)


print("Vector length:", len(sample_vector))
print("First 5 values of the vector:", sample_vector[:5])

Vector length: 1536
First 5 values of the vector: [0.034759521484375, 0.046630859375, 0.08050537109375, 0.0299072265625, -0.0430908203125]


## The Vector 

A vector store is a database for embeddings.

We will use FAISS or we can use ChromaDb.

FAISS helps us search similar chunks quickly.

Flow:

Chunks → Embeddings → FAISS Vector Store

In [26]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(
    documents=chunks,
    embedding=embeddings
)

print("Vector store created successfully.")

Vector store created successfully.


## Retriever

A Retriever finds the most relevant chunks for a user quesiton.

Example:

User asks:

"What problem does dotCom-seccrets solve?"

Retriever searches the vector store and finds related PDF chunks

In [27]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3} #Retrieve the top 3 most relevant chunks. These chunks will be passed to the LLM as context.
    #This reduces hallucination because the model answers based on retrieved content.
)

question = "What are the key strategies mentioned in the document for building a successful online business?"

relevant_chunks = retriever.invoke(question)

print(f"Number of relevant documents retrieved: {len(relevant_chunks)}")

for i, doc in enumerate(relevant_chunks, start=1):
    print(f"\nRelevant Document {i}:")
    print(f"Content: {doc.page_content[:500]}...")
    print(f"Metadata: {doc.metadata}")

Number of relevant documents retrieved: 3

Relevant Document 1:
Content: that these strategies work for both online and oﬄine businesses in just 
about any industry you can think of. Throughout the book, I’ll share 
examples of these diﬀerent types of businesses so you can see how each 
strategy could work in any market....
Metadata: {'producer': 'calibre (5.22.1) [https://calibre-ebook.com]', 'creator': 'calibre (5.22.1) [https://calibre-ebook.com]', 'creationdate': '2015-01-06T17:12:30-07:00', 'author': 'Russell Brunson', 'ebx_publisher': 'Morgan James Publishing', 'gts_pdfxconformance': 'PDF/X-1a:2001', 'gts_pdfxversion': 'PDF/X-1:2001', 'keywords': 'Business & Economics, entrepreneurship, Sales & Selling, General, E-Commerce, Digital Marketing', 'moddate': '2021-07-03T08:38:33-06:00', 'title': 'Dotcom Secrets: The Underground Playbook for Growing Your Company Online', 'trapped': 'False', 'source': 'data/dotcom-secrets.pdf', 'total_pages': 274, 'page': 26, 'page_label': '8'}

Releva

#  Build Final PDF Question Answering Chain

Now we combine everything:

PDF → Loader → Documents → Splitter → Chunks → Embeddings → Vector Store → Retriever → Prompt → LLM → Answer

This is called RAG:

Retrieval-Augmented Generation.

In [28]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

rag_prompt = ChatPromptTemplate.from_template(
    """
    You are a helpful AI teaching assistant.

    Answer the user's question using ONLY the given context.

    If the answer is not available in the context, say:
    "I could not find this in the provided PDF."

    Keep the answer simple, practical, and classroom-friendly.

    Context:
    {context}

    Question:
    {question}

    Answer:
    """
)

rag_chain = rag_prompt | llm | StrOutputParser()

def format_docs(docs):
    formatted_text = []
    for i, doc in enumerate(docs, start=1):
        page = doc.metadata.get("page", "unknown")
        source = doc.metadata.get("source", "unknown")
        formatted_text.append(
            f"Source {i} | Page: {page} | File: {source}\n{doc.page_content}"
        )
    return "\n\n".join(formatted_text)

def ask_pdf(question: str):
    relevant_docs = retriever.invoke(question)
    context = format_docs(relevant_docs)

    answer = rag_chain.invoke({
        "context": context,
        "question": question
    })

    print("=" * 80)
    print("QUESTION:")
    print(question)
    print("\nANSWER:")
    print(answer)

    print("\nSOURCES USED:")
    for doc in relevant_docs:
        print(
            f"- Page: {doc.metadata.get('page', 'unknown')} | "
            f"Source: {doc.metadata.get('source', 'unknown')}"
        )
    print("=" * 80)

In [29]:
ask_pdf("What is the core idea of a value ladder?")

QUESTION:
What is the core idea of a value ladder?

ANSWER:
The core idea of a value ladder is to create a structured range of offers that provide increasing levels of value and service, allowing you to charge more as customers ascend the ladder. It helps in guiding customers from lower-priced offerings to premium services, maximizing both value for the customer and revenue for the business.

SOURCES USED:
- Page: 49 | Source: data/dotcom-secrets.pdf
- Page: 43 | Source: data/dotcom-secrets.pdf
- Page: 41 | Source: data/dotcom-secrets.pdf


In [30]:
ask_pdf("What is the core idea of a communication over landing pages?")

QUESTION:
What is the core idea of a communication over landing pages?

ANSWER:
The core idea of communication over landing pages is to tailor the message based on the type of traffic (hot, warm, or cold) that arrives at the page. Each group requires special treatment and individualized communication to effectively engage them.

SOURCES USED:
- Page: 122 | Source: data/dotcom-secrets.pdf
- Page: 139 | Source: data/dotcom-secrets.pdf
- Page: 136 | Source: data/dotcom-secrets.pdf


PDF File
   ↓
PyPDFLoader
   ↓
Documents
   ↓
RecursiveCharacterTextSplitter
   ↓
Chunks
   ↓
OpenAI Embeddings
   ↓
FAISS Vector Store
   ↓
Retriever
   ↓
Relevant Context
   ↓
Prompt Template
   ↓
ChatOpenAI
   ↓
Answer

Common Errors and Fixes

## Error 1: API key not found

Solution:

Check your `.env` file.

It should contain:

OPENAI_API_KEY=your_api_key_here

## Error 2: PDF not found

Solution:

Keep your PDF in the same folder as this notebook.

Or update this line:

pdf_path = Path("your_file_name.pdf")

## Error 3: Module not found

Solution:

Run the install command again:

pip install -U openai langchain langchain-openai langchain-community langchain-text-splitters faiss-cpu pypdf python-dotenv pydantic

## Error 4: Wrong model name

Solution:

Use a valid model name in `.env`.

Example:

OPENAI_MODEL=gpt-4.1-mini

## Assinments 1: 

# 19. Student Assignments

## Assignment 1

Change the PDF and build your own PDF assistant.

Examples:

- HR policy assistant
- Resume assistant
- Legal document assistant
- Course notes assistant
- Finance report assistant

## Assignment 2

Modify the prompt so that the answer always comes in this format:

1. Short Answer
2. Detailed Explanation
3. Example
4. Source Page

## Assignment 3

Create a structured output version where the answer returns:

- answer
- confidence
- source_pages
- missing_information

## Assignment 4

Save the FAISS vector store locally and load it again without recreating embeddings every time.

so Today we learned:

- How to call OpenAI directly
- How to call OpenAI using LangChain
- How to create prompt templates
- How to generate structured output
- How to load a PDF
- How to split documents
- How to create embeddings
- How to store chunks in FAISS
- How to retrieve relevant chunks
- How to build a PDF question-answering assistant

## Final Formula

LLM alone = smart reply

LLM + LangChain + Documents = real AI application

## RAG Formula

Retrieve relevant information first.

Then generate answer using that information.